In [1]:
import numpy as np
import pandas as pd
import cooler
from cooltools.lib.numutils import observed_over_expected, adaptive_coarsegrain
from cooltools.lib.numutils import interpolate_bad_singletons, set_diag, interp_nan
from astropy.convolution import Gaussian2DKernel
from astropy.convolution import convolve
from bioframe.io.fileops import read_bigwig

In [2]:
import torch
from pyfaidx import Fasta

In [3]:
flame_df_path = "/scratch1/smaruj/stripepy_stripes/selected_stripes.tsv"

In [4]:
flame_df = pd.read_csv(flame_df_path, sep="\t")

In [5]:
FASTA_FILE = "/project/fudenber_735/genomes/mm10/mm10.fa"
COOL_FILE = "/project/fudenber_735/GEO/Hsieh2019/4DN/mESC_mm10_4DNFILZ1CPT8.mapq_30.2048.cool"

# --- Load Data ---
genome = Fasta(FASTA_FILE)

genome_hic_cool = cooler.Cooler(COOL_FILE)

In [6]:
import random

def one_hot_encode_sequence(sequence_obj):
    sequence = str(sequence_obj).upper()
    base_to_int = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    encoded_sequence = np.array([
        base_to_int.get(base, base_to_int[random.choice("ACGT")]) for base in sequence
    ])

    one_hot_encoded = np.zeros((4, len(encoded_sequence)), dtype=np.float32)
    one_hot_encoded[encoded_sequence, np.arange(len(encoded_sequence))] = 1

    return np.expand_dims(one_hot_encoded, axis=0)

In [7]:
def process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1):
    seq_hic_raw = genome_hic_cool.matrix(balance=True).fetch(mseq_str)
    
    # Check for NaN filtering percentage
    seq_hic_nan = np.isnan(seq_hic_raw)
    num_filtered_bins = np.sum(np.sum(seq_hic_nan, axis=0) == len(seq_hic_nan))
    print("num_filtered_bins:", num_filtered_bins)
    
    if num_filtered_bins > (0.5 * len(seq_hic_nan)):
        print(f"More than 50% bins filtered in {mseq_str}. Check Hi-C data quality.")
        
    # clip first diagonals and high values
    clipval = np.nanmedian(np.diag(seq_hic_raw, diagonal_offset))
    for i in range(-diagonal_offset+1, diagonal_offset):
        set_diag(seq_hic_raw, clipval, i)
    seq_hic_raw = np.clip(seq_hic_raw, 0, clipval)
    seq_hic_raw[seq_hic_nan] = np.nan
    
    # adaptively coarsegrain based on raw counts
    seq_hic_smoothed = adaptive_coarsegrain(
                            seq_hic_raw,
                            genome_hic_cool.matrix(balance=False).fetch(mseq_str),
                            cutoff=2, max_levels=8)
    seq_hic_nan = np.isnan(seq_hic_smoothed)
    
    # local obs/exp
    seq_hic_obsexp = observed_over_expected(seq_hic_smoothed, ~seq_hic_nan)[0]
    
    log_hic_obsexp = np.log(seq_hic_obsexp)
    
    # Apply padding
    if padding > 0:
        log_hic_obsexp = log_hic_obsexp[padding:-padding, padding:-padding]
    
    log_hic_obsexp = interp_nan(log_hic_obsexp)
    for i in range(-diagonal_offset+1, diagonal_offset): set_diag(log_hic_obsexp, 0,i)
    
    kernel = Gaussian2DKernel(x_stddev=kernel_stddev)
    seq_hic = convolve(log_hic_obsexp, kernel)
    
    return seq_hic    

In [8]:
def upper_triangular_to_vector_skip_diagonals(matrix, dim=512, diag=2):
    
    # Extract the upper triangular part excluding the first two diagonals
    upper_triangular_vector = matrix[np.triu_indices(dim, k=diag)]
    
    return upper_triangular_vector

In [9]:
N = 256
diagonal_offset = 2

In [10]:
import matplotlib.pyplot as plt
import seaborn as sns

In [11]:
def plot_map(matrix, vmin=-0.6, vmax=0.6, palette="RdBu_r", width=5, height=5):
    fig, axes = plt.subplots(1, 1, figsize=(width, height))

    sns.heatmap(
        matrix,
        vmin=vmin,
        vmax=vmax,
        cbar=False,
        cmap=palette,
        square=True,
        xticklabels=False,
        yticklabels=False,
        ax=axes
    )

    plt.tight_layout()
    plt.show()

In [13]:
flame_df.columns

Index(['chrom1', 'start1', 'end1', 'chrom2', 'start2', 'end2', 'rel_change',
       'frac_missing_bbox_pm5bins_bins', 'x_length', 'y_length', 'orientation',
       'midpoint', 'window_start', 'window_end'],
      dtype='object')

In [ ]:
pearson_r = []

for i, row in enumerate(flame_df.itertuples(index=False)):
    print("index:", i)
    chrom, start, end = row.chrom, row.window_start, row.window_end
    mseq_str = f"{chrom}:{start}-{end}"
    
    # TARGET
    sequence = genome[chrom][start:end]
    ohe_sequence = one_hot_encode_sequence(sequence)
    matrix = process_hic_matrix(genome_hic_cool, mseq_str, diagonal_offset=2, padding=64, kernel_stddev=1)
    
    